In [ ]:
!pip install qiskit
!pip install qiskit-ibm-runtime
!pip install qiskit_aer
!pip install pylatexenc

Trabajar con dos qubits es muy parecido a trabajar con un qubit. Las diferencias son que:

* Declararemos dos qubits y dos bits al crear el circuito
* Las puertas de dos qubits, como la CNOT, reciben como argumentos los dos qubits sobre los que actúan
* Al medir, podemos medir todos los qubits

Como ejemplo, vamos a crear un circuito para preparar un estado entrelazado (un estado de Bell, en concreto) de la forma $$\frac{1}{\sqrt{2}}(|00>+|11>)$$

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2,2)
qc.h(0)
qc.cx(0,1) # Puerta CNOT o CX. El primer qubit es el control y el segundo es el target
qc.measure(range(2),range(2)) # Medimos todos los qubits y recibimos los resultados en los bits clásicos

qc.draw("mpl")

Utilizamos un simulador para ejecutar el circuito. Utilizamos 10 shots (ejecuciones) y una semilla (1234) para que los resultados sean reproducibles.

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_aer import AerSimulator


sampler = Sampler(AerSimulator(seed_simulator = 1234))
job = sampler.run([qc], shots = 10)
results = job.result()
d = results[0].data.c # Acceso a los valores de las mediciones
print("Número de ejercuciones:", d.num_shots)
print("Estadísticas de las ejecuciones:", d.get_counts())
print("Resultados de las medidas:", d.array)
print("Resultados de las medidas como cadenas de texto:", d.get_bitstrings())


Ahora, vamos a calcular de forma exacta las amplitudes del vector de estado final.

In [ ]:
qc.remove_final_measurements()
from qiskit.quantum_info import Statevector
statevector = Statevector(qc)
print("Amplitudes del vector de estado:", statevector)

##BONUS: EJECUCIÓN EN UN ORDENADOR CUÁNTICO REAL

In [ ]:
mytoken = ""

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_cloud", token=mytoken)

In [ ]:
backend = service.least_busy(simulator=False, operational=True)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw('mpl', idle_wires=False)

In [ ]:
sampler = Sampler(backend)
job = sampler.run([isa_circuit], shots = 1000)
results = job.result()
d = results[0].data.meas # Acceso a los valores de las mediciones
print("Estadísticas de las ejecuciones:", d.get_counts())